In [9]:
# importing the classes and loading the datasets 
from data_representation import DataTable
from predictors import MajorityClassPredictor, NearestNeighborPredictor
from exceptions import OutOfSampleError

historical = DataTable("cancer_historical1.csv")

new_cases = DataTable(
    "cancer_new_cases1.csv",
    reference_data=historical
)

labels = DataTable("cancer_new_cases_labels1.csv")

print("Historical rows:", len(historical.data))
print("New case rows:", len(new_cases.data))
print("Label rows:", len(labels.data))


Historical rows: 599
New case rows: 100
Label rows: 100


In [12]:
# load csv data making sure it loads corretly and the data is floats
print("Column names:")
print(historical.get_column_names())

print("\nFirst historical row:")
print(historical.get_row(0))

print("\nFeature1 type:")
print(type(historical.get(0, "Feature1")))


Column names:
['Feature1', 'Feature2', 'Feature3', 'Feature4', 'Feature5', 'Feature6', 'Feature7', 'Feature8', 'Feature9', 'TumorType']

First historical row:
{'Feature1': 5.0, 'Feature2': 1.0, 'Feature3': 1.0, 'Feature4': 1.0, 'Feature5': 2.0, 'Feature6': 1.0, 'Feature7': 3.0, 'Feature8': 1.0, 'Feature9': 1.0, 'TumorType': 0.0}

Feature1 type:
<class 'float'>


In [13]:
#check that the DataTable methods corretly retrieve indiviudal values, rows
#columns, column names, averages, min and max 
print("Individual value:")
print(historical.get(0, "Feature1"))

print("\nFirst row:")
print(historical.get_row(0))

print("\nFeature1 column:")
print(historical.get_column("Feature1")[:5])

print("\nColumn names:")
print(historical.get_column_names())

print("\nFeature6 average:")
print(historical.get_column_average("Feature6"))

print("\nFeature1 minimum:")
print(historical.get_column_min("Feature1"))

print("\nFeature1 maximum:")
print(historical.get_column_max("Feature1"))


Individual value:
5.0

First row:
{'Feature1': 5.0, 'Feature2': 1.0, 'Feature3': 1.0, 'Feature4': 1.0, 'Feature5': 2.0, 'Feature6': 1.0, 'Feature7': 3.0, 'Feature8': 1.0, 'Feature9': 1.0, 'TumorType': 0.0}

Feature1 column:
[5.0, 5.0, 3.0, 6.0, 4.0]

Column names:
['Feature1', 'Feature2', 'Feature3', 'Feature4', 'Feature5', 'Feature6', 'Feature7', 'Feature8', 'Feature9', 'TumorType']

Feature6 average:
3.777397260273973

Feature1 minimum:
1.0

Feature1 maximum:
10.0


In [14]:
#checks that missing feature vlaues are replaced with mean values 

# Check the imputed historical Feature6 values
feature6_values = historical.get_column("Feature6")

print("Historical Feature6 mean:",
      historical.get_column_average("Feature6"))

print("First 10 Feature6 values:", feature6_values[:10])

# Check for any remaining missing values
missing_values = 0

for row in historical.data:
    for column in historical.get_column_names():
        if row[column] is None:
            missing_values += 1

print("Missing values remaining:", missing_values)
print("Historical row count:", len(historical.data))


Historical Feature6 mean: 3.777397260273973
First 10 Feature6 values: [1.0, 10.0, 2.0, 4.0, 1.0, 10.0, 10.0, 1.0, 1.0, 1.0]
Missing values remaining: 0
Historical row count: 599


In [15]:


# Create a small test dataset with a missing Feature6 value
test_file = "test_missing_value.csv"

with open(test_file, "w") as file:
    file.write(
        "Feature1,Feature2,Feature3,Feature4,Feature5,"
        "Feature6,Feature7,Feature8,Feature9,TumorType\n"
    )
    file.write("5,1,1,1,2,?,3,1,1,0\n")

# Load it using the historical data for imputation
test_data = DataTable(test_file, reference_data=historical)

print("Expected Feature6 mean:",
      historical.get_column_average("Feature6"))

print("Imputed Feature6 value:",
      test_data.get(0, "Feature6"))

print("Test dataset rows:", len(test_data.data))



Expected Feature6 mean: 3.777397260273973
Imputed Feature6 value: 3.777397260273973
Test dataset rows: 1


In [17]:
#checking that missing feature values are replaced with the mean in historical dataset

# Count any remaining missing values
missing_historical = 0
missing_new_cases = 0

for patient in historical.data:
    for column in historical.get_column_names():
        if patient[column] is None:
            missing_historical += 1

for patient in new_cases.data:
    for column in new_cases.get_column_names():
        if patient[column] is None:
            missing_new_cases += 1

print("Missing values in historical data:", missing_historical)
print("Missing values in new cases:", missing_new_cases)

test_data = DataTable("test_missing_value.csv", reference_data=historical)

print("Expected Feature6 mean:",
      historical.get_column_average("Feature6"))
print("Imputed Feature6 value:",
      test_data.get(0, "Feature6"))
print("Test dataset rows:", len(test_data.data))


Missing values in historical data: 0
Missing values in new cases: 0
Expected Feature6 mean: 3.777397260273973
Imputed Feature6 value: 3.777397260273973
Test dataset rows: 1


In [18]:
#multiple missing values and row preservations

# Create a small CSV containing multiple missing values
test_file = "test_multiple_missing.csv"

with open(test_file, "w") as file:
    file.write(
        "Feature1,Feature2,Feature3,Feature4,Feature5,"
        "Feature6,Feature7,Feature8,Feature9,TumorType\n"
    )
    file.write("5,1,1,1,2,?,3,1,1,0\n")
    file.write("6,2,2,2,3,?,4,2,2,1\n")
    file.write("7,3,3,3,4,5,5,3,3,0\n")

# Load using historical feature means
test_data = DataTable(test_file, reference_data=historical)

print("Expected rows: 3")
print("Actual rows:", len(test_data.data))

print("\nFeature1 values:", test_data.get_column("Feature1"))
print("Feature6 values:", test_data.get_column("Feature6"))

print("\nFirst row Feature1:", test_data.get(0, "Feature1"))
print("Second row Feature1:", test_data.get(1, "Feature1"))
print("Third row Feature1:", test_data.get(2, "Feature1"))

# Check for remaining missing values
missing_values = 0

for row in test_data.data:
    for column in test_data.get_column_names():
        if row[column] is None:
            missing_values += 1

print("\nMissing values remaining:", missing_values)


Expected rows: 3
Actual rows: 3

Feature1 values: [5.0, 6.0, 7.0]
Feature6 values: [3.777397260273973, 3.777397260273973, 5.0]

First row Feature1: 5.0
Second row Feature1: 6.0
Third row Feature1: 7.0

Missing values remaining: 0


In [19]:
#checking that the class predictor ids the most common tumor class 
majority_predictor = MajorityClassPredictor(historical)

tumor_types = historical.get_column("TumorType")

benign_count = tumor_types.count(0.0)
malignant_count = tumor_types.count(1.0)

print("Benign cases:", benign_count)
print("Malignant cases:", malignant_count)
print("Majority class:", majority_predictor.majority_class)

patient = new_cases.get_row(0)

print("Prediction for first new case:",
      majority_predictor.predict(patient))


Benign cases: 379
Malignant cases: 220
Majority class: 0
Prediction for first new case: 0


In [20]:
# nearest neighbour prediction test


nearest_predictor = NearestNeighborPredictor(historical)

patient = new_cases.get_row(0)
closest_patient = None
smallest_distance = None

for historical_patient in historical.data:
    current_distance = nearest_predictor.distance(
        patient, historical_patient
    )

    if (smallest_distance is None or
            current_distance < smallest_distance):
        smallest_distance = current_distance
        closest_patient = historical_patient

print("Distance to closest historical case:", smallest_distance)
print("Closest case tumor type:", closest_patient["TumorType"])
print("Nearest Neighbor prediction:",
      nearest_predictor.predict(patient))


Distance to closest historical case: 1.7320508075688772
Closest case tumor type: 0.0
Nearest Neighbor prediction: 0


In [21]:
# out of range feature error test 

patient = new_cases.get_row(0).copy()

# Set Feature1 above the historical maximum
patient["Feature1"] = historical.get_column_max("Feature1") + 100

try:
    nearest_predictor.predict(patient)
    print("Test failed: no error was raised")
except OutOfSampleError as error:
    print("Test passed: out-of-range error caught")
    print(error)


Test passed: out-of-range error caught
Feature1 with value 110.0 is outside historical range [1.0, 10.0]


In [22]:
# many out of range features 

patient = new_cases.get_row(0).copy()

# Make two features invalid
patient["Feature1"] = historical.get_column_max("Feature1") + 100
patient["Feature2"] = historical.get_column_max("Feature2") + 100

try:
    nearest_predictor.predict(patient)
    print("Test failed: no error was raised")
except OutOfSampleError as error:
    print("Test passed: first out-of-range error caught")
    print(error)


Test passed: first out-of-range error caught
Feature1 with value 110.0 is outside historical range [1.0, 10.0]


In [23]:
# predection accuracy using percentage 

majority_predictor = MajorityClassPredictor(historical)
nearest_predictor = NearestNeighborPredictor(historical)

true_labels = labels.get_column("TumorType")

majority_correct = 0
nearest_correct = 0

for i in range(len(new_cases.data)):
    patient = new_cases.get_row(i)
    true_label = true_labels[i]

    if majority_predictor.predict(patient) == true_label:
        majority_correct += 1

    if nearest_predictor.predict(patient) == true_label:
        nearest_correct += 1

total_cases = len(true_labels)

majority_accuracy = majority_correct / total_cases * 100
nearest_accuracy = nearest_correct / total_cases * 100

print("Majority Class Predictor")
print("Correct:", majority_correct, "out of", total_cases)
print("Accuracy:", majority_accuracy, "%")

print("\nNearest Neighbor Predictor")
print("Correct:", nearest_correct, "out of", total_cases)
print("Accuracy:", nearest_accuracy, "%")


Majority Class Predictor
Correct: 79 out of 100
Accuracy: 79.0 %

Nearest Neighbor Predictor
Correct: 98 out of 100
Accuracy: 98.0 %
